# Iris Flower Classification: Guided Analysis

This notebook loads the Iris dataset included with scikit-learn, explores its measurements, compares five classifiers, and evaluates the selected model. Run cells from top to bottom. No CSV download is needed.

## 1. Import the tools

Pandas organizes the measurements in a table. Seaborn and Matplotlib make plots. Scikit-learn provides the dataset, split, classifiers, and evaluation metrics.

In [ ]:
import pandas as pd

import matplotlib.pyplot as plt

import seaborn as sns



from sklearn.calibration import CalibratedClassifierCV

from sklearn.datasets import load_iris

from sklearn.ensemble import RandomForestClassifier

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

from sklearn.model_selection import train_test_split

from sklearn.neighbors import KNeighborsClassifier

from sklearn.pipeline import Pipeline

from sklearn.preprocessing import StandardScaler

from sklearn.svm import SVC

from sklearn.tree import DecisionTreeClassifier



RANDOM_STATE = 42

FEATURE_NAMES = ["sepal length", "sepal width", "petal length", "petal width"]

sns.set_theme(style="whitegrid")

## 2. Load and inspect the data

Each row represents one flower. The four numeric columns are features, and `species` is the class we want to predict. The target starts as numbers in scikit-learn, so this cell maps them to readable species labels.

In [ ]:
iris = load_iris()
class_names = [f"Iris-{name}" for name in iris.target_names]
data = pd.DataFrame(iris.data, columns=FEATURE_NAMES)
data["species"] = [class_names[target] for target in iris.target]

print("First five rows:")
display(data.head())
print("Shape (rows, columns):", data.shape)
print("Columns:", data.columns.tolist())
print("Data types:")
display(data.dtypes)
print("Missing values:")
display(data.isna().sum())
print("Descriptive statistics:")
display(data[FEATURE_NAMES].describe())
print("Samples per species:")
display(data["species"].value_counts().sort_index())

## 3. Visualize the measurements

The distribution chart checks class balance. The pair plot compares every feature pair; the two scatter plots focus on sepal and petal measurements. Colors identify species.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.countplot(data=data, x="species", hue="species", legend=False, ax=axes[0])
axes[0].set_title("Species Distribution")
axes[0].tick_params(axis="x", rotation=15)
sns.scatterplot(data=data, x="sepal length", y="sepal width", hue="species", ax=axes[1])
axes[1].set_title("Sepal Length vs Sepal Width")
plt.tight_layout()
plt.show()

sns.scatterplot(data=data, x="petal length", y="petal width", hue="species", s=65)
plt.title("Petal Length vs Petal Width")
plt.show()

sns.pairplot(data, hue="species", corner=True, diag_kind="hist")
plt.show()

## 4. Separate features and target, then split

`X` contains the four measurements and `y` contains species labels. We reserve 20% of rows for testing. Stratification keeps the three species represented evenly in both sets; `random_state` makes the split repeatable. Scaling is included inside each applicable model pipeline, so it is learned only from training data and does not leak information from the test set.

In [ ]:
X = data[FEATURE_NAMES]
y = data["species"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

## 5. Train and compare five models

StandardScaler puts measurement columns on a comparable scale for Logistic Regression, KNN, and SVM. Tree models split by feature thresholds, so scaling is not needed for them. Accuracy is the fraction of test predictions that are correct; each classification report also shows precision, recall, and F1-score per species.

In [ ]:
models = {

    "Logistic Regression": Pipeline([

        ("scaler", StandardScaler()),

        ("classifier", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),

    ]),

    "K-Nearest Neighbors": Pipeline([

        ("scaler", StandardScaler()), ("classifier", KNeighborsClassifier(n_neighbors=5))

    ]),

    "Decision Tree": DecisionTreeClassifier(random_state=RANDOM_STATE),

    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE),

    "Support Vector Machine": Pipeline([

        ("scaler", StandardScaler()),

        ("classifier", CalibratedClassifierCV(SVC(), cv=5)),

    ]),

}



scores = []

for name, model in models.items():

    model.fit(X_train, y_train)

    predictions = model.predict(X_test)

    accuracy = accuracy_score(y_test, predictions)

    scores.append({"Model": name, "Accuracy": accuracy, "Macro F1": f1_score(y_test, predictions, average="macro")})

    print(f"\n{name}: accuracy={accuracy:.2%}")

    print(classification_report(y_test, predictions, labels=class_names, zero_division=0))



comparison = pd.DataFrame(scores).sort_values(["Accuracy", "Macro F1"], ascending=False)

display(comparison.style.format({"Accuracy": "{:.2%}", "Macro F1": "{:.3f}"}))

## 6. Evaluate the selected classifier

We select by test accuracy, using macro F1 to break ties. A confusion matrix has actual species as rows and predictions as columns. Correct predictions appear along its diagonal; other cells are errors. Since the same holdout set is used for comparison and selection, these scores are an educational estimate, not a separate final benchmark.

In [ ]:
best_name = comparison.iloc[0]["Model"]
best_model = models[best_name]
best_predictions = best_model.predict(X_test)
print("Selected model:", best_name)
print(f"Accuracy: {accuracy_score(y_test, best_predictions):.2%}")
print("Classification report:")
print(classification_report(y_test, best_predictions, labels=class_names, zero_division=0))

matrix = confusion_matrix(y_test, best_predictions, labels=class_names)
sns.heatmap(matrix, annot=True, fmt="d", cmap="YlGnBu",
            xticklabels=class_names, yticklabels=class_names, cbar=False)
plt.title(f"Confusion Matrix: {best_name}")
plt.xlabel("Predicted species")
plt.ylabel("Actual species")
plt.tight_layout()
plt.show()

## Next step

Run `python src/train_model.py` from the project root to save the selected trained model bundle and reproducible PNG visualizations for the terminal predictor and Streamlit app.